# Classes, code-review-graph alone: spring-petclinic (java)
Baseline. How many of the reference's classes does code-review-graph store as `Class` nodes?
Match = same file, name and start line. graph.db is opened read-only.

In [1]:
import pandas as pd, class_lab as L
pd.set_option('display.max_colwidth', 100); pd.set_option('display.width', 250)
APP, LANG = 'spring-petclinic', 'java'
REPO = L.ROOT / 'apps' / APP
DB = L.ROOT / 'results' / APP / 'runs/iter-002_2026-11-17_2008/crg/graph.db'
files = L.repo_files(REPO, LANG)
print(LANG, 'files:', len(files))

java files: 50


## 1. Reference: javac (`java-ref/ListClasses.java`)
Named classes are expected. Anonymous classes have no name and are not expected.

In [2]:
ref = L.reference(REPO, LANG, files)
print('parse errors:', ref.error.notna().sum())
ref = ref[ref.error.isna()]
print(ref.kind.value_counts().to_string())
named = ref[ref.kind != 'anonymous'].reset_index(drop=True)
print('expected named classes:', len(named))

parse errors: 0
kind
class        47
interface     3
anonymous     3
expected named classes: 50


## 2. code-review-graph Class nodes

In [3]:
crg = L.crg_classes(DB, REPO, LANG, files)
print('CRG Class nodes:', len(crg))

CRG Class nodes: 50


## 3. Score

In [4]:
c, c_miss, c_extra = L.score(named, crg)
pd.Series(c).to_frame('code-review-graph')

,code-review-graph
expected,50.0
found,50.0
found %,100.0
same end line,50.0
missed,0.0
extra,0.0


## 4. What failed
**Missed** = in the reference, no CRG node with the same file, name and start line.
**Extra** = CRG node with no match in the reference.

In [5]:
c_miss.assign(source=[L.source_line(REPO, f, l) for f, l in zip(c_miss.file, c_miss.line)])

,file,name,line,end,kind,source


In [6]:
c_extra.assign(source=[L.source_line(REPO, f, l) for f, l in zip(c_extra.file, c_extra.line_start)])

,file,name,line_start,line_end,source


## 5. False positives
**False positive** = a CRG Class node whose name is not a class in that file in the reference (any line).
A wrong start line is not a false positive: the class exists.

In [7]:
c_fp = L.false_positives(named, crg, REPO)
print('false positives (this language):', len(c_fp))
c_fp

false positives (this language): 0


,file,name,line_start,line_end,source


In [8]:
other = L.crg_other_files(DB, REPO, files)
print('CRG Class nodes in other file types:', len(other))
display(other.language.value_counts())
other.head(20)

CRG Class nodes in other file types: 21


language
sql    21
Name: count, dtype: int64

,file,name,line_start,language
0,src/main/resources/db/h2/schema.sql,vets,10,sql
1,src/main/resources/db/h2/schema.sql,specialties,17,sql
2,src/main/resources/db/h2/schema.sql,vet_specialties,23,sql
3,src/main/resources/db/h2/schema.sql,types,30,sql
4,src/main/resources/db/h2/schema.sql,owners,36,sql
5,src/main/resources/db/h2/schema.sql,pets,46,sql
6,src/main/resources/db/h2/schema.sql,visits,58,sql
7,src/main/resources/db/mysql/schema.sql,vets,1,sql
8,src/main/resources/db/mysql/schema.sql,specialties,8,sql
9,src/main/resources/db/mysql/schema.sql,vet_specialties,14,sql
